In [3]:
import math
import warnings
from pathlib import Path
from plotly import graph_objects as go
import numpy as np
import numpy.typing as npt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
import random
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Makes cuDNN deterministic (slight perf cost, irrelevant on MPS)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
DATA_DIR = Path("../data")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else 
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)

## Data Loading

In [6]:
class WindowDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: list[str],
        seq_len: int = 20,
        target_col: str = "target",
    ):
        self.seq_len = seq_len
        self.samples: list[tuple[npt.NDArray[np.float32], np.float32]] = []

        for ticker, group in df.groupby("ticker"):
            group = group.sort_index()
            X = group[feature_cols].values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)

            mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
            X = X[mask]
            y = y[mask]

            for i in range(seq_len, len(group)):
                window = X[i - seq_len : i]
                target = y[i]
                self.samples.append((window, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.tensor(y)

In [7]:
def make_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    seq_len: int = 20,
    batch_size: int = 64,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    kw = dict(feature_cols=feature_cols, seq_len=seq_len)
    train_ds = WindowDataset(train_df, **kw)
    val_ds = WindowDataset(val_df, **kw)
    test_ds = WindowDataset(test_df, **kw)

    loader_kw = dict(batch_size=batch_size, num_workers=0)  # num_workers=0 on MPS
    return (
        DataLoader(train_ds, shuffle=True, **loader_kw),
        DataLoader(val_ds, shuffle=False, **loader_kw),
        DataLoader(test_ds, shuffle=False, **loader_kw),
    )

## Model

In [8]:
class DirectionalHuberLoss(nn.Module):
    """Huber + bonus when sign(pred) == sign(target)."""
    def __init__(self, delta=1.0, dir_weight=0.3):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()
        # Reward correct direction, penalize wrong direction
        sign_match = (preds.sign() == targets.sign()).float()
        dir_loss = 1.0 - sign_match.mean()
        return huber + self.dir_weight * dir_loss

In [9]:
class ResidualGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2,
                 linear_hidden=64, dropout=0.2):
        super().__init__()

        # Main recurrent path
        self.gru = nn.GRU(
            input_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # Skip path: projects raw input to same dim as GRU output
        # Operates on the LAST timestep, matching what GRU outputs
        self.skip = nn.Linear(input_size, hidden_size)

        # Shared head that processes the combined representation
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, linear_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(linear_hidden, 1),
        )

    def forward(self, x):
        # x: (batch, seq_len, input_size)

        gru_out, h_n = self.gru(x)
        gru_last = gru_out[:, -1, :]        # (batch, hidden_size)

        skip_out = self.skip(x[:, -1, :])   # (batch, hidden_size)

        combined = gru_last + skip_out       # residual addition

        out = self.head(combined).squeeze(-1)
        return out, h_n

## Model torchure

In [10]:
def train_one_epoch(
    model: ResidualGRU,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    clip_grad: float = 1.0,
) -> float:
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds, _ = model(x)
        loss = criterion(preds, y)
        loss.backward()

        if clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        total_loss += loss.item() * len(y)

    return total_loss / len(loader.dataset)

In [11]:
@torch.no_grad()
def evaluate(
    model: ResidualGRU,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    """
    Returns a dict with loss, MAE, and directional accuracy.

    Directional accuracy: fraction of predictions where sign(pred) == sign(true).
    This is the headline metric for a return-prediction model –
    a coin-flip baseline is 0.50.
    """
    model.eval()
    all_preds, all_targets = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        preds, _ = model(x)
        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())

    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)

    loss = criterion(preds, targets).item()
    mae = (preds - targets).abs().mean().item()
    dir_acc = (preds.sign() == targets.sign()).float().mean().item()

    return {"loss": loss, "mae": mae, "dir_acc": dir_acc}

In [12]:
def train(
    model:        ResidualGRU,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    device:       torch.device,
    lr:           float = 1e-4,
    epochs:       int   = 50,
    patience:     int   = 10,
    huber_delta:  float = 1.0,
) ->ResidualGRU:
    """
    Full training loop with:
     - Huber loss (delta=1.0 on z-scored targets is well calibrated)
     - AdamW + cosine LR schedule
     - Early stopping on val loss
     - W&B logging
    """
    criterion = DirectionalHuberLoss(delta=huber_delta)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float("inf")
    patience_ctr  = 0
    best_state    = None

    logs = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        logs.append({
            "epoch":          epoch,
            "train/loss":     train_loss,
            "val/loss":       val_metrics["loss"],
            "val/mae":        val_metrics["mae"],
            "val/dir_acc":    val_metrics["dir_acc"],
                "lr":             scheduler.get_last_lr()[0],
            })

        print(
            f"Epoch {epoch:3d} | "
            f"train_loss={train_loss:.4f}  "
            f"val_loss={val_metrics['loss']:.4f}  "
            f"val_dir_acc={val_metrics['dir_acc']:.3f}"
        )

        # Early stopping
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stop at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model


In [13]:
train_df = pd.read_csv("data/train_features.csv", index_col=0, parse_dates=True)
val_df   = pd.read_csv("data/val_features.csv",   index_col=0, parse_dates=True)
test_df  = pd.read_csv("data/test_features.csv",  index_col=0, parse_dates=True)


# Combine train + val for CV; keep test held out as final evaluation
cv_df   = pd.concat([train_df, val_df]).sort_index()
print(f"CV pool:  {len(cv_df):,} rows  |  "
      f"{cv_df.index.min().date()} → {cv_df.index.max().date()}")
print(f"Test set: {len(test_df):,} rows (held out)")
feature_cols = [
    "dev_100",
    "bull_regime",
    "log_close_return_1",
    "z_range",
    "z_volume_z20",
    "log_low_return_1",
    "log_volume",
    "z_macd_hist",
]
print(f"Features ({len(feature_cols)}): {feature_cols[:5]} ...")

CV pool:  12,005 rows  |  2019-07-18 → 2025-07-08
Test set: 1,464 rows (held out)
Features (8): ['dev_100', 'bull_regime', 'log_close_return_1', 'z_range', 'z_volume_z20'] ...


In [14]:
from timeseries_cv import cross_validate, summarize_cv, compare_strategies
from timeseries_cv import expanding_window_folds, sliding_window_folds

huber_standard = nn.HuberLoss(delta=1.01)

print("EXPANDING:")
for f in expanding_window_folds(cv_df, n_folds=5, min_train_frac=0.40, val_frac=0.10):
    print(f"  Fold {f.fold_idx}: train {f.train_start.date()} → {f.train_end.date()} "
          f"| val {f.val_start.date()} → {f.val_end.date()}")
    targets  = cv_df.loc[f.val_start : f.val_end, "target"]
    loss = huber_standard(
        torch.zeros_like(torch.from_numpy(targets.values)),  # dummy preds
        torch.from_numpy(targets.values)
    ).item()
    print(f"    Huber loss on val set (dummy preds): {loss:.4f}")

print("\nSLIDING:")
for f in sliding_window_folds(cv_df, n_folds=5, train_frac=0.40, val_frac=0.10):
    print(f"  Fold {f.fold_idx}: train {f.train_start.date()} → {f.train_end.date()} "
          f"| val {f.val_start.date()} → {f.val_end.date()}")

EXPANDING:
  Fold 0: train 2019-07-18 → 2021-12-06 | val 2021-12-07 → 2022-07-12
    Huber loss on val set (dummy preds): 0.4246
  Fold 1: train 2019-07-18 → 2022-09-05 | val 2022-09-06 → 2023-04-11
    Huber loss on val set (dummy preds): 0.3375
  Fold 2: train 2019-07-18 → 2023-06-05 | val 2023-06-06 → 2024-01-09
    Huber loss on val set (dummy preds): 0.4249
  Fold 3: train 2019-07-18 → 2024-03-04 | val 2024-03-05 → 2024-10-08
    Huber loss on val set (dummy preds): 0.3999
  Fold 4: train 2019-07-18 → 2024-12-02 | val 2024-12-03 → 2025-07-08
    Huber loss on val set (dummy preds): 0.3662

SLIDING:
  Fold 0: train 2019-07-18 → 2021-12-06 | val 2021-12-07 → 2022-07-12
  Fold 1: train 2020-04-16 → 2022-09-05 | val 2022-09-06 → 2023-04-11
  Fold 2: train 2021-01-14 → 2023-06-05 | val 2023-06-06 → 2024-01-09
  Fold 3: train 2021-10-14 → 2024-03-04 | val 2024-03-05 → 2024-10-08
  Fold 4: train 2022-07-14 → 2024-12-02 | val 2024-12-03 → 2025-07-08


In [18]:
SEQ_LEN = 10

def model_factory():
    torch.manual_seed(SEED)
    return ResidualGRU(
    input_size=len(feature_cols), hidden_size=128,
    num_layers=2, linear_hidden=128, dropout=0.1,
).to(DEVICE)

huber_loss = nn.HuberLoss(delta=1.01)

shared_params = dict(
    full_df=cv_df, feature_cols=feature_cols,
    model_factory=model_factory, dataset_cls=WindowDataset,
    n_folds=5, train_frac=0.40, val_frac=0.10,
    seq_len=SEQ_LEN, batch_size=64, lr=3e-4,
    epochs=300, patience=20, huber_delta=1.01, device=DEVICE,
    weight_decay=1e-2, criterion=huber_loss
)

exp_results = cross_validate(**shared_params, strategy="expanding")
exp_df = summarize_cv(exp_results)

# sli_results = cross_validate(**shared_params, strategy="sliding")
# sli_df = summarize_cv(sli_results)


  [EXPANDING] Fold 0  |  train 2019-07-18 → 2021-12-06  |  val 2021-12-07 → 2022-07-12
  Epoch   1  train_loss=0.5930  val_loss=0.4848  dir_acc=0.518
  Epoch   2  train_loss=0.4565  val_loss=0.4260  dir_acc=0.497
  Epoch   3  train_loss=0.4177  val_loss=0.4254  dir_acc=0.508
  Epoch   4  train_loss=0.4061  val_loss=0.4288  dir_acc=0.494
  Epoch   5  train_loss=0.3992  val_loss=0.4360  dir_acc=0.481
  Epoch   6  train_loss=0.3963  val_loss=0.4309  dir_acc=0.478
  Epoch   7  train_loss=0.3907  val_loss=0.4275  dir_acc=0.494
  Epoch   8  train_loss=0.3892  val_loss=0.4332  dir_acc=0.498
  Epoch   9  train_loss=0.3883  val_loss=0.4283  dir_acc=0.502
  Epoch  10  train_loss=0.3883  val_loss=0.4338  dir_acc=0.478
  Epoch  11  train_loss=0.3857  val_loss=0.4338  dir_acc=0.485
  Epoch  12  train_loss=0.3847  val_loss=0.4276  dir_acc=0.496
  Epoch  13  train_loss=0.3828  val_loss=0.4288  dir_acc=0.498
  Epoch  14  train_loss=0.3824  val_loss=0.4295  dir_acc=0.500


KeyboardInterrupt: 

In [ ]:
exp_results 

NameError: name 'exp_results' is not defined

In [ ]:
# sli_results = cross_validate(**shared_params, strategy="sliding")
# sli_df = summarize_cv(sli_results)
# comp_df = compare_strategies(exp_results, sli_results)

In [ ]:
import plotly.graph_objects as go

fold_labels = [f"Fold {r.fold_idx}" for r in exp_results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.val_dir_acc for r in exp_results],
    name="Expanding", marker_color="#636efa", opacity=0.8,
))
# fig.add_trace(go.Bar(
#     x=fold_labels,
#     y=[r.val_dir_acc for r in sli_results],
#     name="Sliding", marker_color="#ef553b", opacity=0.8,
# ))
fig.add_hline(y=0.50, line_dash="dash", line_color="white",
              annotation_text="coin flip")
fig.update_layout(
    template="plotly_dark",
    title="Expanding vs Sliding Window — Dir. Accuracy per Fold",
    yaxis_title="Dir. Accuracy",
    yaxis_range=[0.40, 0.65],
    barmode="group",
)
fig.show()

In [ ]:
cv_df.to_csv("data/gru_cv_pool.csv")